# EoMT trên BTXRD với siêu tham số điều chỉnh theo dữ liệu (Google Colab)

Cùng cấu trúc và quy trình với `btxrd_eomt_colab.ipynb`: huấn luyện **EoMT** (Encoder-only Mask Transformer, CVPR 2025) của dự án trên bộ X-quang u xương **BTXRD**, một lần forward cho mỗi ảnh vừa **phát hiện** có u hay không, vừa **phân đoạn** vùng u, vừa **phân loại** u lành / ác. Khác biệt duy nhất: **các siêu tham số được chọn lại theo đặc điểm của dữ liệu** (bảng dưới).

- Ảnh có u: mỗi polygon là một mask, lớp `benign` hoặc `malignant`. Ảnh bình thường: 0 mask, mọi query học dự đoán lớp "no object" có sẵn của EoMT.
- Dữ liệu: dataset Kaggle [`nguyetanhmai/btxrd-full-split`](https://www.kaggle.com/datasets/nguyetanhmai/btxrd-full-split), 3.726 ảnh đã chia train : val : test = 80 : 15 : 5.

### Đặc điểm dữ liệu

| Split | Bình thường | U lành | U ác | Lành : ác |
|---|---|---|---|---|
| train | 1.503 (50,4%) | 1.212 (40,7%) | 265 (8,9%) | 4,6 : 1 |
| val | 282 (50,4%) | 227 (40,6%) | 50 (8,9%) | 4,5 : 1 |
| test | 94 (50,3%) | 76 (40,6%) | 17 (9,1%) | 4,5 : 1 |

- **Một nửa số ảnh không có u**, và trong ảnh có u thì **u ác chỉ chiếm 18%**.
- **U lành rất nhỏ:** diện tích mask trung vị chỉ 0,52% ảnh (10% số ca dưới 0,05%), trong khi u ác trung vị 3,1% (10% số ca dưới 1,1%). Ở đầu vào 640 px, u lành trung vị rộng khoảng 40 px (2,5 patch 16 px), còn 10% u nhỏ nhất chỉ khoảng 12 px, nhỏ hơn 1 patch. Mask được dự đoán ở 1/4 độ phân giải, nên các u này chỉ còn khoảng 3 px.
- 87% ảnh có u chỉ có 1 polygon (tối đa 9). Cạnh dài ảnh gốc trung vị 2.021 px; bản cache trong notebook thu về ≤ 1.024 px.

### Siêu tham số

| Tham số | `btxrd_eomt_colab` | Notebook này | Lý do |
|---|---|---|---|
| `img_size` | 640 | **896** | U lành nhỏ: u trung vị rộng 40 → 56 px, u nhỏ nhất (10%) 12 → 17 px; lưới token 40×40 → 56×56. Ảnh cache 1.024 px nên vẫn đủ chi tiết. |
| Checkpoint COCO | dùng nguyên | **nội suy `pos_embed` 40×40 → 56×56** | Bắt buộc khi đổi `img_size`: checkpoint COCO huấn luyện ở 640; cách nội suy giống timm khi đổi kích thước ảnh. |
| `scale_range` (augmentation) | [0.5, 1.5] | **[0.8, 1.6]** | Thu nhỏ 0,5× làm u lành nhỏ gần như biến mất; phóng to tới 1,6× (cắt ngẫu nhiên vẫn giữ u) giúp học chi tiết u nhỏ. |
| Batch hiệu dụng | 8 | 8 | Giữ. Batch nhỏ hơn × tích lũy gradient tùy bộ nhớ GPU (T4: 2 × 4). |
| `max_epochs` | 100 | **60** | Lịch lr poly tính theo `max_epochs`: với 100 epoch, lr lúc early stopping (khoảng epoch 40–50) vẫn còn khoảng 60%, mô hình không có giai đoạn lr nhỏ để tinh chỉnh. Với 60 epoch, lr giảm về 0 ngay trong pha chọn mô hình (epoch 34–60). |
| Early stopping `patience` | 5 | **8** | Val chỉ có 50 u ác, nên AUROC lành/ác (1/3 của `val_score`) dao động mạnh giữa các epoch; patience 5 dễ dừng sớm vì nhiễu. |
| `num_queries` | 200 | **50** | 200 query là cho COCO (tới khoảng 100 vật thể / ảnh), trong khi BTXRD tối đa 9 u / ảnh, 87% chỉ có 1 u. Điểm phát hiện là max P(u) trên mọi query, nên càng nhiều query thì một ảnh bình thường càng dễ có một query "báo nhầm". Mỗi query cũng được ghép với u thật thường hơn (tỉ lệ dương 1/200 → 1/50). Embedding query lấy 50 query đầu của checkpoint COCO; 50 vẫn gấp hơn 5 lần số u nhiều nhất. |
| `lr` | 1e-4 | **5e-5** | Batch hiệu dụng 8, bằng nửa batch 16 của công thức EoMT gốc (README), nên lr giảm một nửa theo quy tắc tỉ lệ tuyến tính. Tập train chỉ 2.980 ảnh (265 u ác), lr lớn làm head khớp quá nhanh với dữ liệu train. |
| LLRD | 0,8 | **0,9** | X-quang xám khác xa ảnh tự nhiên của DINOv2 / COCO, nên các block đầu (cạnh, kết cấu xương) cần thích nghi nhiều hơn. Với 0,8, block đầu chỉ nhận 0,8¹¹ ≈ 9% lr; với 0,9 là 31%. Kết hợp với lr 5e-5: head và block cuối học chậm hơn một nửa, block đầu học nhanh hơn khoảng 1,8 lần, tức lr dàn đều hơn giữa các tầng. |
| Weight decay | 0,05 | **0,1** | AdamW suy giảm trọng số theo tích lr × wd; tăng wd lên gấp đôi khi lr giảm một nửa giữ nguyên mức điều chuẩn của dự án, không để nó yếu đi trên một tập dữ liệu nhỏ. |
| Warmup | [500, 1000] bước | **[1, 2] epoch** (≈ [372, 745] bước) | Giai đoạn 1 chỉ huấn luyện head (class head khởi tạo mới, bộ query bị cắt), backbone đứng yên; giai đoạn 2 tăng dần lr của backbone. lr đã nhỏ hơn nên warmup ngắn hơn là đủ: backbone đạt lr đầy đủ ở epoch 3 thay vì khoảng epoch 4, còn nhiều thời gian thích nghi với X-quang trước khi mask annealing bắt đầu (epoch khoảng 7). Tính theo epoch nên không đổi khi số GPU thay đổi. |
| Hệ số loss: mask (BCE) / Dice / class | 5 / 5 / 2 | **2 / 5 / 4** | 2 trong 3 thành phần của `val_score` (AUROC phát hiện, AUROC lành/ác) chỉ phụ thuộc class logits, và phân biệt lành / ác (u ác 9%) là phần khó nhất, nên tăng trọng số class. Dice chuẩn hóa theo kích thước từng mask, coi u lành rất nhỏ ngang u lớn và chính là chỉ số đánh giá, nên giữ 5 và giảm BCE (theo điểm, thiên về vùng u lớn) xuống 2. Hệ số này cũng là chi phí ghép cặp Hungarian. Tổng trọng số gần như không đổi (12 → 11). |
| Hệ số no-object | 0,1 | 0,1 | Giữ: một nửa số ảnh không có u, mọi query phải học "no object" trên các ảnh đó; giảm từ 200 xuống 50 query đã làm tỉ lệ dương / âm tăng khoảng 4 lần. |
| `balanced_sampling` | true | true | Giữ: mỗi batch lấy đều bình thường / lành / ác (u ác được lặp khoảng 3,7×/epoch); không có nó, u ác chỉ chiếm 9% tín hiệu học. |
| `tumor_score_thresh`, `malignant_thresh` | 0,5 / 0,5 | **chọn trên val** (Youden J) | Balanced sampling làm mô hình thấy tỉ lệ u : bình thường 2 : 1 và lành : ác 1 : 1, khác thực tế 1 : 1 và 4,6 : 1, nên P(u), P(ác) bị lệch và ngưỡng 0,5 không còn phù hợp. Ngưỡng tối đa hóa độ nhạy + độ đặc hiệu trên val (mục 5) được dùng cho test. |
| Còn lại | | giữ nguyên | EoMT-B DINOv2 fine-tune từ COCO panoptic, poly 0.9, số điểm lấy mẫu cho mask loss, mask annealing theo lịch 40 epoch, tiêu chí `val_score` = (AUROC phát hiện + Dice + AUROC lành/ác) / 3, vốn không phụ thuộc ngưỡng và không bị lệch bởi mất cân bằng lớp. |

**Thời gian:** ở 896 px, mỗi epoch chậm hơn 640 px khoảng 2 lần. Trên T4 cần vài phiên Colab; khi phiên bị ngắt thì *Run all* lại để huấn luyện tiếp từ `last.ckpt`. Đặt `IMG_SIZE = 640` để quay về tốc độ cũ (các thay đổi khác vẫn giữ).

**Chạy:** *Runtime → Change runtime type → GPU (T4 trở lên)*, rồi *Run all*.
- Dataset Kaggle private: thêm `KAGGLE_USERNAME`, `KAGGLE_KEY` vào 🔑 *Secrets* của Colab (bật *Notebook access*).
- Mô hình tốt nhất và kết quả test lưu ở **`MyDrive/eomt/new_eomt/`**, tách khỏi kết quả của notebook gốc. Colab ngắt giữa chừng thì *Run all* lại: huấn luyện tiếp từ `last.ckpt` (trạng thái early stopping cũng được khôi phục); huấn luyện đã xong thì bỏ qua bước huấn luyện.

## 1. Cấu hình

In [3]:
import torch
print(torch.__version__)   # có '+cpu' -> bản CPU; có '+cu12x' -> bản GPU
!nvidia-smi

2.14.1+cpu
/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import os
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

SMOKE_TEST = False  # True = 1 epoch x 2 batches to check the pipeline (written to MyDrive/eomt/new_eomt/smoke_test)

# --- Paths ---
WORK_DIR = Path("/content") if ON_COLAB else Path.cwd().parent
DRIVE_DIR = Path("/content/drive/MyDrive/eomt/new_eomt") if ON_COLAB else WORK_DIR / "eomt_results" / "new_eomt"
OUT_DIR = DRIVE_DIR / "smoke_test" if SMOKE_TEST else DRIVE_DIR
REPO_URL = "https://github.com/67htb5vn/eomt.git"
REPO_BRANCH = "master"
REPO_DIR = WORK_DIR / "eomt" if ON_COLAB else Path.cwd()

# --- Data ---
KAGGLE_DATASET = "nguyetanhmai/btxrd-full-split"
DATA_DIR = WORK_DIR / "btxrd_cache"  # resized copy written by tools/prepare_btxrd.py (local disk, fast)

# --- Model / training: BASE_CONFIG, with the hyperparameters below chosen for BTXRD (see the table above) ---
BASE_CONFIG = "configs/dinov2/btxrd/instance/eomt_base_640.yaml"  # EoMT-B (DINOv2)
COCO_CKPT_REPO = "tue-mps/coco_panoptic_eomt_base_640_2x"  # fine-tune from EoMT-B COCO panoptic (README)
IMG_SIZE = 896  # input size (project: 640); small benign tumors need more pixels
SCALE_RANGE = [0.8, 1.6]  # scale jitter of the training augmentation (project: [0.5, 1.5])
BALANCED_SAMPLING = True  # normal / benign / malignant sampled equally in training (as the project)
MAX_EPOCHS = 60  # also the length of the poly lr schedule; early stopping may end training before
EARLY_STOP_PATIENCE = 8  # epochs without improvement of the validation criterion
ANNEAL_EPOCHS = 40  # mask annealing spread over the project's 40 epochs (masked attention off at epoch 34)
# Optimization / loss chosen for BTXRD (project value in brackets, see the table above)
NUM_QUERIES = 50  # [200] at most 9 tumors per image; the first 50 COCO query embeddings are kept
LR = 5e-5  # [1e-4] effective batch 8 = half the batch 16 of the EoMT recipe; small dataset
LLRD = 0.9  # [0.8] X-ray domain: early ViT blocks adapt more (block 0 gets 0.9^11 = 31% of LR, was 9%)
WEIGHT_DECAY = 0.1  # [0.05] same decoupled decay LR x WD as the project; small dataset
WARMUP_EPOCHS = (1, 2)  # [500, 1000 steps ~ 1.3, 2.7 epochs] heads alone, then backbone ramp-up
MASK_COEF, DICE_COEF, CLASS_COEF = 2.0, 5.0, 4.0  # [5, 5, 2] loss (and matching cost) weights
NO_OBJECT_COEF = 0.1  # [0.1] kept
TUNE_THRESHOLDS = True  # choose tumor / malignant thresholds on val (False = 0.5 / 0.5)
RUN_NAME = f"btxrd_instance_eomt_base_{IMG_SIZE}_tuned"
NUM_WORKERS = min(4, os.cpu_count() or 2)
RESUME = True  # continue from last.ckpt if it exists

CKPT_DIR = OUT_DIR / "checkpoints"
RUN_CONFIG = OUT_DIR / "run.yaml"  # overrides of BASE_CONFIG for this run
THRESH_CONFIG = OUT_DIR / "thresholds.yaml"  # tumor / malignant thresholds chosen on val
TEST_CONFIG = OUT_DIR / "test.yaml"  # overrides for evaluation
TRAIN_DONE = OUT_DIR / "train_done.txt"  # written when training ends (early stop or max_epochs)
BEST_WEIGHTS = OUT_DIR / "best.bin"
RESULTS_DIR = OUT_DIR / "results"
VAL_RESULTS = RESULTS_DIR / "val"
TEST_RESULTS = RESULTS_DIR / "test"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Results ->", OUT_DIR)

## 2. Cài đặt: mã nguồn dự án và thư viện

In [ ]:
if not (REPO_DIR / "main.py").exists():
    !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements_btxrd.txt
# pycocotools 2.0.8 (pinned in the cloned requirements) has no Python 3.13 wheel and its source
# build fails with NumPy 2 ("module 'numpy' has no attribute 'NPY_OWNDATA'"); 2.0.11 fixes it
!pip install -q "pycocotools==2.0.11"
sys.path.insert(0, str(REPO_DIR))  # project modules (its datasets/ before Hugging Face `datasets`)

import torch

NUM_GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])

## 3. Chuẩn bị dữ liệu

`tools/prepare_btxrd.py` tải dataset `nguyetanhmai/btxrd-full-split` từ Kaggle (giữ nguyên cách chia trong `split.csv`), và ghi một bản ảnh đã thu nhỏ (cạnh dài ≤ 1024 px, polygon co theo) vào `DATA_DIR`. Chạy lại sẽ bỏ qua nếu đã chuẩn bị xong.

In [ ]:
!python tools/prepare_btxrd.py --kaggle-dataset {KAGGLE_DATASET} --out {DATA_DIR}

## 4. Huấn luyện (tối đa 60 epoch, early stopping)

Lệnh của dự án: `main.py fit -c <config gốc> -c <config lần chạy>`. File `run.yaml` ghi đè:
- Siêu tham số mới: `img_size: [896, 896]`, `scale_range: [0.8, 1.6]`, `max_epochs: 60`, early stopping `patience: 8`, `num_q: 50`, `lr: 5e-5`, `llrd: 0.9`, `weight_decay: 0.1`, `warmup_steps` = [1, 2] epoch, hệ số loss mask / Dice / class = 2 / 5 / 4.
- Phần phụ thuộc máy: đường dẫn dữ liệu, số GPU, batch × tích lũy gradient (batch hiệu dụng 8), logger CSV, nơi lưu checkpoint.
- Checkpoint COCO để fine-tune (`ckpt_path`, `load_ckpt_class_head: false` như README). Checkpoint này huấn luyện ở 640 px với 200 query, nên ma trận vị trí `pos_embed` (40×40 vị trí patch) được nội suy bicubic sang lưới 56×56 bằng `timm.layers.resample_abs_pos_embed`, đúng như timm làm khi tạo ViT ở kích thước khác, và embedding query `network.q.weight` giữ 50 hàng đầu. Các trọng số còn lại không phụ thuộc kích thước ảnh hay số query.

Những gì không ghi đè (poly 0.9, số điểm lấy mẫu của mask loss, ...) lấy nguyên từ config gốc.

**Mask annealing là gì.** EoMT huấn luyện với *masked attention*: ở 3 block cuối, mỗi query chỉ nhìn vào vùng mask nó đang dự đoán. Khi suy luận, dự án tắt masked attention (`masked_attn_enabled: false`) cho nhanh. Để mô hình quen chạy không có mask, trong lúc huấn luyện xác suất áp mask của từng block giảm dần từ 1 về 0 (annealing). `MaskClassificationBTXRD.on_fit_start` tính lịch này theo tỉ lệ tổng số bước, giống các config COCO / ADE20K: block *k* bắt đầu giảm ở 1/6, 2.5/6, 4/6 thời gian huấn luyện và mỗi block giảm trong 1/6. Masked attention tắt hẳn ở **5/6** thời gian huấn luyện, phần còn lại chạy giống lúc suy luận.

| | Annealing kết thúc | Chọn mô hình + early stopping |
|---|---|---|
| Dự án gốc, 40 epoch | epoch 34 (5/6 × 40) | không có, dùng mô hình epoch 40 |
| **Notebook này: 60 epoch, `ANNEAL_EPOCHS = 40`** | **epoch 34, như dự án gốc** | **từ epoch 34 đến tối đa 60** |
| `ANNEAL_EPOCHS = MAX_EPOCHS` (tỉ lệ theo 60 epoch) | epoch 50 | từ epoch 50: early stopping chỉ còn tác dụng trong 10 epoch cuối |

Trước khi annealing kết thúc, điểm val được đo khi masked attention vẫn còn bật một phần, nên không phản ánh mô hình lúc test. Vì vậy hai callback trong `training/btxrd_callbacks.py` (có sẵn trong repo đã clone, cell dưới chỉ import để kiểm tra) bỏ qua kết quả val của các epoch đó:
- `ModelCheckpointFrom`: là `ModelCheckpoint` của config gốc (giữ checkpoint có `metrics/val_score` cao nhất + `last.ckpt` để resume, lưu ở `MyDrive/eomt/new_eomt/checkpoints`), nhưng chỉ chọn checkpoint tốt nhất trong các epoch ≥ epoch annealing kết thúc.
- `EarlyStoppingFrom`: là `EarlyStopping(monitor="metrics/val_score", mode="max", patience=8)`, đếm patience từ epoch đó. Huấn luyện dừng khi `val_score` không tăng trong 8 epoch liên tiếp, hoặc khi đủ 60 epoch.

Lịch annealing (`attn_mask_annealing_start_steps` / `end_steps`) được tính bằng đúng công thức của `on_fit_start` với số bước của `ANNEAL_EPOCHS` epoch, rồi ghi vào `run.yaml`.

In [ ]:
from training.btxrd_callbacks import EarlyStoppingFrom, ModelCheckpointFrom  # code lives in the repo

print("Callbacks loaded from", ModelCheckpointFrom.__module__)

In [ ]:
from training.mask_classification_btxrd import MaskClassificationBTXRD  # code lives in the repo

assert "val_mask_ap" in MaskClassificationBTXRD.__init__.__code__.co_varnames,     "Repo clone is outdated: delete REPO_DIR and rerun section 2"
print("MaskClassificationBTXRD loaded from", MaskClassificationBTXRD.__module__)

In [ ]:
import copy
import math

import yaml
from huggingface_hub import hf_hub_download
from timm.layers import resample_abs_pos_embed

from datasets.btxrd import BTXRDDataset, load_image_types

base_cfg = yaml.safe_load(Path(BASE_CONFIG).read_text())
BATCH_SIZE = base_cfg["data"]["init_args"]["batch_size"]  # 8 in the project config
encoder_args = base_cfg["model"]["init_args"]["network"]["init_args"]["encoder"].get("init_args", {})
PATCH_SIZE = encoder_args.get("patch_size", 16)  # models.vit.ViT default

# Same effective batch on any GPU: the largest micro-batch that fits x gradient accumulation.
# Rough memory need: ~2.5 GB per image at 640 px, growing with the number of tokens (img_size^2);
# keep 15% of the GPU memory free as a margin (an A100 40 GB then runs 4 x 2 at 896 px)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30 if NUM_GPUS else 0
gb_per_image = 2.5 * (IMG_SIZE / 640) ** 2
MICRO_BATCH = max(b for b in (8, 4, 2, 1) if b <= BATCH_SIZE and (b == 1 or b * gb_per_image <= 0.85 * gpu_mem_gb))
ACCUMULATE = BATCH_SIZE // MICRO_BATCH
print(f"GPU memory {gpu_mem_gb:.0f} GB, {IMG_SIZE} px -> batch {MICRO_BATCH} x accumulate {ACCUMULATE} = {BATCH_SIZE}")


def coco_ckpt_for(img_size, num_q):
    """EoMT-B COCO panoptic weights (640 px, 200 queries) adapted to img_size and num_q:
    pos_embed resampled to the img_size grid, query embeddings cut to the first num_q."""
    src = hf_hub_download(COCO_CKPT_REPO, "pytorch_model.bin")
    if img_size == 640 and num_q == 200:
        return src
    dst = WORK_DIR / f"coco_panoptic_eomt_base_{img_size}_q{num_q}.bin"
    if dst.exists():
        return dst
    state = torch.load(src, map_location="cpu", weights_only=True)
    state = state.get("state_dict", state)
    q_keys = [k for k in state if k.endswith("network.q.weight")]
    assert q_keys, "query embedding not found in the COCO checkpoint"
    for key in q_keys:
        print(f"{key}: {tuple(state[key].shape)} -> ", end="")
        state[key] = state[key][:num_q].clone()
        print(tuple(state[key].shape))
    for key in [k for k in state if k.endswith("pos_embed")]:
        pos = state[key]
        old = math.isqrt(pos.shape[1])
        prefix = pos.shape[1] - old * old  # tokens without a grid position (none for DINOv2 reg4 in timm)
        new = img_size // PATCH_SIZE
        state[key] = resample_abs_pos_embed(pos, new_size=(new, new), old_size=(old, old), num_prefix_tokens=prefix)
        print(f"{key}: {tuple(pos.shape)} -> {tuple(state[key].shape)}")
    torch.save(state, dst)
    return dst


coco_ckpt = coco_ckpt_for(IMG_SIZE, NUM_QUERIES)

# Mask annealing: formula of MaskClassificationBTXRD.on_fit_start applied to the optimizer
# steps of ANNEAL_EPOCHS epochs (the project's 40) instead of all MAX_EPOCHS
n_train = len(BTXRDDataset(str(DATA_DIR), "train", load_image_types(str(DATA_DIR))))
batches_per_epoch = math.ceil(n_train / max(1, NUM_GPUS)) // MICRO_BATCH  # train loader: drop_last=True
steps_per_epoch = math.ceil(batches_per_epoch / ACCUMULATE)
num_blocks = base_cfg["model"]["init_args"]["network"]["init_args"]["num_blocks"]
unit = steps_per_epoch * ANNEAL_EPOCHS / 6
anneal_start = [round(unit + k * 3 * unit / max(1, num_blocks - 1)) for k in range(num_blocks)]
anneal_end = [round(s + unit) for s in anneal_start]
# Warm-up in optimizer steps: WARMUP_EPOCHS[0] epochs for the heads (backbone frozen), then the
# backbone ramps up over WARMUP_EPOCHS[1] epochs (TwoStageWarmupPolySchedule)
warmup_steps = [round(e * steps_per_epoch) for e in WARMUP_EPOCHS]
print(f"warm-up steps {warmup_steps} (project: [500, 1000])")
# First epoch (0-based) whose validation runs with masked attention fully off
SELECT_FROM_EPOCH = 0 if SMOKE_TEST else math.ceil(anneal_end[-1] / steps_per_epoch) - 1
print(f"{n_train} train images, {steps_per_epoch} steps/epoch | mask annealing starts {anneal_start}, "
      f"ends {anneal_end} (epoch {SELECT_FROM_EPOCH + 1}) -> best model + early stopping from epoch "
      f"{SELECT_FROM_EPOCH + 1} on, max {MAX_EPOCHS} epochs")

# The project's checkpoint callback (saved to Drive) and early stopping on the same criterion,
# both counting only the epochs after mask annealing
checkpoint_cb = copy.deepcopy(base_cfg["trainer"]["callbacks"][0])
checkpoint_cb["class_path"] = "training.btxrd_callbacks.ModelCheckpointFrom"
checkpoint_cb["init_args"].update(dirpath=str(CKPT_DIR), start_epoch=SELECT_FROM_EPOCH)
MONITOR = checkpoint_cb["init_args"]["monitor"]  # metrics/val_score
early_stop_cb = {
    "class_path": "training.btxrd_callbacks.EarlyStoppingFrom",
    "init_args": {
        "monitor": MONITOR,
        "mode": checkpoint_cb["init_args"]["mode"],  # max
        "patience": EARLY_STOP_PATIENCE,
        "start_epoch": SELECT_FROM_EPOCH,
        "verbose": True,
    },
}

run_cfg = {
    "trainer": {
        "max_epochs": 1 if SMOKE_TEST else MAX_EPOCHS,
        "devices": max(1, NUM_GPUS),
        "accelerator": "gpu" if NUM_GPUS else "cpu",
        "strategy": "ddp" if NUM_GPUS > 1 else "auto",
        "accumulate_grad_batches": ACCUMULATE,
        "log_every_n_steps": 50,
        "logger": {
            "class_path": "lightning.pytorch.loggers.CSVLogger",
            "init_args": {"save_dir": str(OUT_DIR / "logs"), "name": RUN_NAME},
        },
        "callbacks": [checkpoint_cb, early_stop_cb],
    },
    "model": {
        "init_args": {
            "ckpt_path": str(coco_ckpt),
            "load_ckpt_class_head": False,  # 133 COCO classes -> 2 BTXRD classes
            "attn_mask_annealing_start_steps": anneal_start,
            "attn_mask_annealing_end_steps": anneal_end,
            "val_mask_ap": False,  # mask AP only in main.py test (see above)
            "lr": LR,
            "llrd": LLRD,
            "weight_decay": WEIGHT_DECAY,
            "warmup_steps": warmup_steps,
            "mask_coefficient": MASK_COEF,
            "dice_coefficient": DICE_COEF,
            "class_coefficient": CLASS_COEF,
            "no_object_coefficient": NO_OBJECT_COEF,
            "eval_top_k_instances": min(100, NUM_QUERIES * 2),  # mask AP: top-k of queries x 2 classes
            "network": {"init_args": {"num_q": NUM_QUERIES}},
        }
    },
    "data": {
        "init_args": {
            "path": str(DATA_DIR),
            "batch_size": MICRO_BATCH,
            "num_workers": NUM_WORKERS,
            "img_size": [IMG_SIZE, IMG_SIZE],  # linked to the model, network and encoder by main.py
            "scale_range": SCALE_RANGE,
            "balanced_sampling": BALANCED_SAMPLING,
        }
    },
}
if not NUM_GPUS:
    run_cfg["trainer"]["precision"] = "32-true"
if SMOKE_TEST:
    run_cfg["trainer"].update(limit_train_batches=2, limit_val_batches=2, num_sanity_val_steps=0)
RUN_CONFIG.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
print(RUN_CONFIG.read_text())

if TRAIN_DONE.exists():
    print("Training already finished:", TRAIN_DONE.read_text())
else:
    last_ckpt = CKPT_DIR / "last.ckpt"
    resume = f"--ckpt_path {last_ckpt}" if RESUME and not SMOKE_TEST and last_ckpt.exists() else ""
    !python main.py fit -c {BASE_CONFIG} -c {RUN_CONFIG} --compile_disabled {resume}
    if _exit_code != 0:
        raise RuntimeError("main.py fit failed, see the log above")
    TRAIN_DONE.write_text(f"best checkpoint: {sorted(p.name for p in CKPT_DIR.glob('best-*.ckpt'))}\n")
    print("Training finished:", TRAIN_DONE.read_text())

### Diễn biến tiêu chí trên tập val

Đọc log CSV (gộp các lần resume): `val_score` và các thành phần theo từng epoch (đánh số từ 1). Cột `phase`: `annealing` = masked attention còn bật một phần, không dùng để chọn mô hình; `selection` = chạy như lúc suy luận, dùng để chọn mô hình tốt nhất (tô xanh) và đếm patience.

In [ ]:
import pandas as pd

cols = {
    MONITOR: "val_score",
    "metrics/val_det_auroc": "det_auroc",
    "metrics/val_dice": "dice",
    "metrics/val_cls_auroc": "cls_auroc",
    "metrics/val_img3_acc": "img3_acc",
}
logs = sorted((OUT_DIR / "logs" / RUN_NAME).glob("version_*/metrics.csv"), key=os.path.getmtime)
hist = pd.concat([pd.read_csv(p) for p in logs])
val_hist = (
    hist.dropna(subset=[MONITOR]).groupby("epoch")[list(cols)].last().rename(columns=cols)
)
val_hist.insert(0, "phase", ["annealing" if e < SELECT_FROM_EPOCH else "selection" for e in val_hist.index])
val_hist.index += 1  # 1-based epochs, as in the text
selection = val_hist[val_hist.phase == "selection"]
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
val_hist.to_csv(RESULTS_DIR / "val_history.csv")
num_cols = list(cols.values())
styled = val_hist.style.format("{:.4f}", subset=num_cols)
if len(selection):
    best_epoch = int(selection.val_score.idxmax())
    styled = styled.apply(lambda r: ["background-color: #c6efce" if r.name == best_epoch else "" for _ in r], axis=1)
display(styled)
if len(selection):
    print(
        f"Best epoch (chosen among epochs >= {SELECT_FROM_EPOCH + 1}): {best_epoch} "
        f"(val_score {selection.val_score.max():.4f}) | last epoch: {int(val_hist.index.max())} | "
        f"epochs without improvement at the end: {int(val_hist.index.max()) - best_epoch} "
        f"(patience {EARLY_STOP_PATIENCE}, max {MAX_EPOCHS} epochs)"
    )
else:
    print(f"Mask annealing not finished yet: model selection starts at epoch {SELECT_FROM_EPOCH + 1}")

## 5. Xuất trọng số tốt nhất và chọn ngưỡng trên tập val

Chuyển checkpoint tốt nhất (theo `metrics/val_score` trên val) thành state dict `MyDrive/eomt/new_eomt/best.bin`, định dạng README dùng cho `--model.ckpt_path`.

**Chọn ngưỡng.** Mô hình tốt nhất dự đoán toàn bộ tập val bằng `tools/predict_btxrd.py`. Hai ngưỡng sau được chọn bằng Youden J (tối đa độ nhạy + độ đặc hiệu, tức tối đa balanced accuracy) rồi ghi vào `thresholds.yaml`, file này dùng cho mọi bước đánh giá trên test:
- `tumor_score_thresh`: ảnh có u nếu P(u) > ngưỡng; xét trên mọi ảnh val.
- `malignant_thresh`: u ác nếu P(ác | u) ≥ ngưỡng; xét trên các ảnh val có u.

Balanced accuracy không phụ thuộc tỉ lệ lớp, nên lớp thiểu số (u ác) được coi trọng ngang lớp đa số. Tập test không tham gia vào việc chọn ngưỡng.

In [ ]:
!python tools/export_weights.py --ckpt-dir {CKPT_DIR} --out {BEST_WEIGHTS}

In [ ]:
import numpy as np
from IPython.display import Image, Markdown, display
from sklearn.metrics import balanced_accuracy_score, roc_curve


def youden_threshold(y_true, score, strict):
    """Threshold maximizing sensitivity + specificity; strict=True for a `score > t` rule."""
    fpr, tpr, thr = roc_curve(y_true, score)
    ok = np.isfinite(thr) & (thr > 0.01) & (thr < 0.99)
    best = thr[ok][np.argmax((tpr - fpr)[ok])]
    return float(best - 1e-6) if strict else float(best)  # roc_curve predicts positive for score >= t


def summarize(df, tumor_t, mal_t):
    """Image-level metrics of predictions.csv rows for given thresholds."""
    has_tumor, tumor_rows = df.label > 0, df[df.label > 0]
    det = df.p_tumor > tumor_t
    mal = tumor_rows.p_malignant >= mal_t
    img3 = np.where(det, 1 + (df.p_malignant >= mal_t), 0)
    is_mal = tumor_rows.label == 2
    return {
        "det_sensitivity": det[has_tumor].mean(), "det_specificity": (~det[~has_tumor]).mean(),
        "det_balanced_acc": balanced_accuracy_score(has_tumor, det),
        "cls_sensitivity (malignant)": mal[is_mal].mean(), "cls_specificity (benign)": (~mal[~is_mal]).mean(),
        "cls_balanced_acc": balanced_accuracy_score(is_mal, mal),
        "img3_acc": (img3 == df.label).mean(), "img3_balanced_acc": balanced_accuracy_score(df.label, img3),
    }


TUMOR_THRESH = MALIGNANT_THRESH = 0.5
if TUNE_THRESHOLDS:
    !python tools/predict_btxrd.py -c {BASE_CONFIG} -c {RUN_CONFIG} --weights {BEST_WEIGHTS} --data-path {DATA_DIR} --split val --out {VAL_RESULTS} --num-vis 0
    val_pred = pd.read_csv(VAL_RESULTS / "predictions.csv")
    val_tumor = val_pred[val_pred.label > 0]
    TUMOR_THRESH = youden_threshold(val_pred.label > 0, val_pred.p_tumor, strict=True)
    MALIGNANT_THRESH = youden_threshold(val_tumor.label == 2, val_tumor.p_malignant, strict=False)
    display(Markdown(f"**Ngưỡng chọn trên val:** u: `{TUMOR_THRESH:.3f}`, ác: `{MALIGNANT_THRESH:.3f}`"))
    display(pd.DataFrame({
        "val, ngưỡng 0.5 / 0.5": summarize(val_pred, 0.5, 0.5),
        "val, ngưỡng đã chọn": summarize(val_pred, TUMOR_THRESH, MALIGNANT_THRESH),
    }).style.format("{:.4f}"))

THRESH_CONFIG.write_text(yaml.safe_dump(
    {"model": {"init_args": {"tumor_score_thresh": TUMOR_THRESH, "malignant_thresh": MALIGNANT_THRESH}}},
    sort_keys=False,
))
print(THRESH_CONFIG.read_text())

## 6. Đánh giá trên tập test (CLI của dự án)

Như phần *Evaluating* của README: tắt masked attention khi suy luận và nạp trọng số tốt nhất qua `model.ckpt_path`, chạy trên 1 GPU, với ngưỡng trong `thresholds.yaml`. In ra mask AP và các chỉ số phát hiện / phân đoạn / lành-ác / 3 lớp; bảng chỉ số được chép vào `MyDrive/eomt/new_eomt/results/test_cli_metrics.csv`.

In [ ]:
import shutil

test_cfg = {
    "trainer": {
        "devices": 1,
        "strategy": "auto",
        "logger": {
            "class_path": "lightning.pytorch.loggers.CSVLogger",
            "init_args": {"save_dir": str(OUT_DIR / "logs"), "name": f"{RUN_NAME}_test"},
        },
    },
    "model": {
        "init_args": {
            "ckpt_path": str(BEST_WEIGHTS),
            "load_ckpt_class_head": True,
            "network": {"init_args": {"masked_attn_enabled": False}},
        }
    },
}
TEST_CONFIG.write_text(yaml.safe_dump(test_cfg, sort_keys=False))
!python main.py test -c {BASE_CONFIG} -c {RUN_CONFIG} -c {THRESH_CONFIG} -c {TEST_CONFIG}

test_logs = sorted((OUT_DIR / "logs" / f"{RUN_NAME}_test").glob("version_*/metrics.csv"), key=os.path.getmtime)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(test_logs[-1], RESULTS_DIR / "test_cli_metrics.csv")
test_cli = pd.read_csv(test_logs[-1]).ffill().iloc[-1].dropna()
keys = ["score", "det_auroc", "det_sensitivity", "det_specificity", "dice", "iou",
        "cls_auroc", "cls_balanced_acc", "img3_acc", "img3_balanced_acc"]
display(test_cli[[f"metrics/test_{k}" for k in keys if f"metrics/test_{k}" in test_cli]].to_frame("test"))

## 7. Inference và đánh giá theo từng ảnh trên tập test

`tools/predict_btxrd.py` dự đoán từng ảnh (có u hay không, vùng u, lành / ác), ghi vào `MyDrive/eomt/new_eomt/results/test/`: `predictions.csv`, báo cáo phân loại cho từng phần (`report.txt`, `metrics.json`), confusion matrix, ROC và hình minh họa `vis/`. Dự đoán dùng ngưỡng trong `thresholds.yaml`. Riêng phần lành / ác của `report.txt` / `metrics.json`, script luôn tính ở ngưỡng 0,5, nên bảng thứ hai tính lại với ngưỡng đã chọn (`metrics_tuned_thresholds.csv`).

In [ ]:
from IPython.display import Image, Markdown, display

!python tools/predict_btxrd.py -c {BASE_CONFIG} -c {RUN_CONFIG} -c {THRESH_CONFIG} --weights {BEST_WEIGHTS} --data-path {DATA_DIR} --split test --out {TEST_RESULTS}

import json

test_metrics = json.loads((TEST_RESULTS / "metrics.json").read_text())
display(Markdown("**Chỉ số trên tập test (theo từng ảnh, `metrics.json`)**"))
display(pd.Series(test_metrics, name="test").to_frame())

# report.txt / metrics.json score benign vs malignant at a fixed 0.5; these use the val-chosen thresholds
test_pred = pd.read_csv(TEST_RESULTS / "predictions.csv")
display(Markdown(f"**Chỉ số trên tập test với ngưỡng chọn trên val** (u `{TUMOR_THRESH:.3f}`, ác `{MALIGNANT_THRESH:.3f}`)"))
test_summary = pd.Series(summarize(test_pred, TUMOR_THRESH, MALIGNANT_THRESH), name="test")
test_summary.to_frame().to_csv(TEST_RESULTS / "metrics_tuned_thresholds.csv")
display(test_summary.to_frame().style.format("{:.4f}"))
display(Image(str(TEST_RESULTS / "confusion_matrix.png")), Image(str(TEST_RESULTS / "roc.png")))
display(Markdown("**Vài ảnh test ngẫu nhiên**"))
for p in sorted((TEST_RESULTS / "vis").glob("random_*.png"))[:3]:
    display(Image(str(p)))

## 8. Một số kết quả trên tập test: dự đoán đúng và các trường hợp thất bại

Từ `predictions.csv`, chọn ca tiêu biểu cho từng nhóm, chạy lại mô hình tốt nhất trên các ảnh đó (hàm `build_model`, `predict`, `save_figure` của `tools/predict_btxrd.py`) và hiển thị: ảnh | ground truth | dự đoán (xanh lá = lành, đỏ = ác, viền vàng = vùng u thật).

| Nhóm | Điều kiện | Ca được chọn |
|---|---|---|
| Đúng: bình thường | bình thường → bình thường | `P(tumor)` thấp nhất |
| Đúng: u lành / u ác | đúng loại u | Dice cao nhất |
| Thất bại: u bị bỏ sót | có u → bình thường | sai tự tin nhất: `P(tumor)` thấp nhất |
| Thất bại: báo động nhầm | bình thường → có u | `P(tumor)` cao nhất |
| Thất bại: nhầm lành / ác | phát hiện có u nhưng sai loại | `P(malignant)` xa nhãn thật nhất |
| Thất bại: phân đoạn kém | phát hiện có u nhưng Dice < 0.5 | Dice thấp nhất |

Danh sách đầy đủ ghi vào `MyDrive/eomt/new_eomt/results/test/examples.csv` (cột `group`, `failure`), hình vào `results/test/examples/`.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "tools"))
from predict_btxrd import build_model, merge, predict, save_figure  # noqa: E402

N_CORRECT = 2  # correct predictions shown per group
N_FAILURE = 3  # failure cases shown per type
LOW_DICE = 0.5

df = pd.read_csv(TEST_RESULTS / "predictions.csv")
tumor, detected = df.label > 0, df.pred > 0
wrong_type_conf = (df.p_malignant - (df.label == 2)).abs()  # distance of P(malignant) to the true label
# key: (title, is failure, rows, sort key, ascending)
groups = {
    "correct_normal": ("Đúng: bình thường", False, (df.label == 0) & (df.pred == 0), df.p_tumor, True),
    "correct_benign": ("Đúng: u lành", False, (df.label == 1) & (df.pred == 1), df.dice, False),
    "correct_malignant": ("Đúng: u ác", False, (df.label == 2) & (df.pred == 2), df.dice, False),
    "missed": ("Thất bại: u bị bỏ sót (có u → bình thường)", True, tumor & ~detected, df.p_tumor, True),
    "false_alarm": ("Thất bại: báo động nhầm (bình thường → có u)", True, ~tumor & detected, df.p_tumor, False),
    "swapped": ("Thất bại: nhầm lành / ác", True, tumor & detected & (df.label != df.pred), wrong_type_conf, False),
    "low_dice": (f"Thất bại: phân đoạn kém (phát hiện có u, Dice < {LOW_DICE})", True,
                 tumor & detected & (df.dice < LOW_DICE), df.dice, True),
}

examples = pd.concat(
    df[rows].assign(group=key, failure=failure, sort_key=sort_key[rows]).sort_values("sort_key", ascending=asc)
    for key, (title, failure, rows, sort_key, asc) in groups.items()
)
examples.to_csv(TEST_RESULTS / "examples.csv", index=False)

summary = pd.DataFrame(
    {"group": [g[0] for g in groups.values()],
     "count": [int(g[2].sum()) for g in groups.values()]},
    index=list(groups),
)
summary["out_of"] = [int((df.label == c).sum()) for c in (0, 1, 2)] + [
    int(tumor.sum()), int((~tumor).sum()), int((tumor & detected).sum()), int((tumor & detected).sum())
]
display(summary)

# Re-run the best model on the cases shown (same read-out as predict_btxrd.py)
config = {}
for path in (BASE_CONFIG, RUN_CONFIG, THRESH_CONFIG):
    config = merge(config, yaml.safe_load(Path(path).read_text()))
device = torch.device("cuda" if NUM_GPUS else "cpu")
model = build_model(config, str(BEST_WEIGHTS), device)
ds = BTXRDDataset(str(DATA_DIR), "test", load_image_types(str(DATA_DIR)))
index_of = {sample[0].stem: i for i, sample in enumerate(ds.samples)}
fig_dir = TEST_RESULTS / "examples"
fig_dir.mkdir(parents=True, exist_ok=True)

show_cols = ["image", "label_name", "pred_name", "p_tumor", "p_malignant", "dice"]
fmt = {"p_tumor": "{:.3f}", "p_malignant": "{:.3f}", "dice": "{:.3f}"}
for key, (title, failure, *_rest) in groups.items():
    rows = examples[examples.group == key].head(N_FAILURE if failure else N_CORRECT)
    display(Markdown(f"### {title}: {int(summary.loc[key, 'count'])} ca"))
    if rows.empty:
        print("Không có ca nào.")
        continue
    display(rows[show_cols].style.format(fmt, na_rep="-").hide(axis="index"))
    for _, row in rows.iterrows():
        img, target = ds[index_of[row.image]]
        pred_mask, *_ = predict(model, img, device)
        fig_path = fig_dir / f"{key}_{row.image}.png"
        save_figure(fig_path, img, target["masks"].bool().any(0), int(target["image_label"]), pred_mask, row)
        display(Image(str(fig_path)))

del model
torch.cuda.empty_cache()

## 9. Dự đoán cho ảnh X-quang mới

Thay `NEW_IMAGES` bằng đường dẫn ảnh (hoặc thư mục ảnh) của bạn.

In [ ]:
NEW_IMAGES = sorted((DATA_DIR / "test" / "images").iterdir())[:3]  # example: 3 test images
NEW_RESULTS = RESULTS_DIR / "new"
image_args = " ".join(map(str, NEW_IMAGES))
!python tools/predict_btxrd.py -c {BASE_CONFIG} -c {RUN_CONFIG} -c {THRESH_CONFIG} --weights {BEST_WEIGHTS} --images {image_args} --out {NEW_RESULTS}

for p in sorted((NEW_RESULTS / "vis").glob("*.png")):
    display(Image(str(p)))

## Kết quả đầu ra (trong `MyDrive/eomt/new_eomt/`)

| File | Nội dung |
|---|---|
| `best.bin` | Trọng số tốt nhất theo `val_score` (state dict), dùng với `--model.ckpt_path` |
| `checkpoints/best-epochXX.ckpt`, `last.ckpt` | Checkpoint Lightning (tốt nhất + cuối, dùng để resume) |
| `run.yaml`, `test.yaml` | Phần ghi đè cấu hình của lần chạy (gồm siêu tham số mới) / lần đánh giá |
| `thresholds.yaml` | Ngưỡng u / ác chọn trên val, dùng cho mọi đánh giá trên test |
| `train_done.txt` | Đánh dấu huấn luyện đã xong (xóa file này để huấn luyện lại) |
| `logs/` | Log CSV của huấn luyện và đánh giá (validation không có mask AP; mask AP chỉ có ở log test) |
| `results/val_history.csv` | `val_score` và các thành phần theo epoch (pha annealing / chọn mô hình) |
| `results/val/` | Dự đoán trên val dùng để chọn ngưỡng (`predictions.csv`, `report.txt`, ...) |
| `results/test_cli_metrics.csv` | Chỉ số test từ `main.py test` |
| `results/test/` | `predictions.csv`, `report.txt`, `metrics.json`, `metrics_tuned_thresholds.csv`, `confusion_matrix.png`, `roc.png`, `vis/` |
| `results/test/examples.csv`, `results/test/examples/` | Ca dự đoán đúng tiêu biểu + các ca thất bại, kèm hình |
| `results/new/` | Dự đoán cho ảnh mới |